# 09주차 · 멀티모달 임베딩

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 텍스트와 이미지를 공동 임베딩 공간에 배치하는 원리를 설명한다.
- CLIP의 쌍대 인코더 구조와 대조학습을 이해한다.
- 무예시 이미지 분류 결과를 정량·정성 평가한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


In [ ]:
from PIL import Image, ImageDraw

def shape_image(shape, color, size=224):
    image = Image.new("RGB", (size, size), "white")
    draw = ImageDraw.Draw(image)
    if shape == "circle": draw.ellipse((45, 45, 179, 179), fill=color)
    if shape == "square": draw.rectangle((45, 45, 179, 179), fill=color)
    if shape == "triangle": draw.polygon([(112, 35), (35, 185), (189, 185)], fill=color)
    return image

color_names = [("blue", "파란"), ("red", "빨간"), ("green", "초록")]
shape_names = [("circle", "원"), ("square", "사각형"), ("triangle", "삼각형")]
examples = [(color, shape, f"{ko_color} {ko_shape}") for color, ko_color in color_names for shape, ko_shape in shape_names]
images = [shape_image(shape, color) for color, shape, _ in examples]
labels = [label for _, _, label in examples]
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax, image, label in zip(axes.ravel(), images, labels):
    ax.imshow(image); ax.set_title(label); ax.axis("off")
plt.show()


## 모델 없이 공동 공간의 원리 확인

다음 one-hot 벡터는 학습된 CLIP 결과가 아니라 **원리 확인용 장난감 공간**이다. 이미지와 텍스트가 같은 색·도형 좌표를 사용하면 내적으로 교차 모달 검색이 가능해진다. 실제 CLIP은 이러한 좌표를 대조학습으로 얻는다.


In [ ]:
color_index = {name: i for i, (name, _) in enumerate(color_names)}
shape_index = {name: i for i, (name, _) in enumerate(shape_names)}

def toy_shared_embedding(color=None, shape=None):
    vector = np.zeros(6, dtype=float)
    if color is not None: vector[color_index[color]] = 1.0
    if shape is not None: vector[3 + shape_index[shape]] = 1.0
    return vector / np.linalg.norm(vector)

image_embeddings = np.vstack([toy_shared_embedding(color, shape) for color, shape, _ in examples])
toy_queries = [("파란 원", "blue", "circle"), ("초록 삼각형", "green", "triangle"), ("빨간 도형", "red", None)]
text_embeddings = np.vstack([toy_shared_embedding(color, shape) for _, color, shape in toy_queries])
toy_scores = image_embeddings @ text_embeddings.T
pd.DataFrame(toy_scores, index=labels, columns=[q[0] for q in toy_queries]).round(3)


## 선택 실습: CLIP

모델은 최초 실행 시 다운로드가 필요하다. 한국어 질의 성능이 낮으면 영문 프롬프트와 다국어 CLIP 모델을 비교하는 연구 질문으로 확장할 수 있다.


In [ ]:
RUN_CLIP = False
if RUN_CLIP:
    import torch
    from transformers import CLIPModel, CLIPProcessor
    model_name = "openai/clip-vit-base-patch32"
    model = CLIPModel.from_pretrained(model_name)
    processor = CLIPProcessor.from_pretrained(model_name)
    prompt_sets = {
        "English": [f"a {color} {shape}" for color, shape, _ in examples],
        "한국어": labels,
    }
    for language, prompts in prompt_sets.items():
        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
        with torch.no_grad():
            outputs = model(**inputs)
        probs = outputs.logits_per_image.softmax(dim=1).cpu().numpy()
        top1_accuracy = float(np.mean(probs.argmax(axis=1) == np.arange(len(images))))
        print(f"{language} prompt top-1 accuracy: {top1_accuracy:.3f}")
        print(pd.DataFrame(probs, index=labels, columns=prompts).round(3).to_string())
else:
    print("CLIP 선택 실습을 건너뜁니다.")


## 학생 활동

- 제공된 9개 도형 중 색상이나 모양 하나를 바꾸고 검색 순위 변화를 기록하라.
- 텍스트 프롬프트의 구체성에 따른 순위 변화를 기록하라.
- 한국어와 영어 프롬프트 성능을 비교하라.
- 사람·직업·성별과 관련된 실제 이미지는 편향 분석 계획 없이 사용하지 않는다.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
